# AquaHealth AI — Layer 3: fold-aware GAN augmentation on Colab GPU

Trains **one class-conditional DCGAN per CV fold on that fold's training rows only** (`data/audit/cv_v2/fold_XX_train.csv`) and writes the fold's synthetic training images. The validation fold and the frozen final test set are runtime-forbidden ids — the GAN aborts if any of them is offered. Everything runs through the repository scripts; nothing is implemented in this notebook.

Order: 1 clone → 2 CUDA/GPU smoke → 3 Drive deliveries → keys → local bundle → 4 integrity (5,942 hashes, frozen-test / fold digests) → 5 persist → **6 GAN smoke (tiny)** → 7 inspect → **8 full 10-fold generation** → 9 verify → 10 export the records for `git`.

The architecture (cDCGAN, 64×64, z=100, Adam 2e-4/(0.5, 0.999), batch 64, 30 epochs, seed 42) is the project's own documented engineering assumption — no supplied paper or document specifies a GAN (`docs/GAN_AUGMENTATION.md` §1). The earlier local MPS fold-1 output is not an official result and is not used.

## 1. Runtime → GPU. Clone `develop`, install the pinned dependencies

In [ ]:
!nvidia-smi
import os, pathlib, subprocess
REPO_URL = 'https://github.com/kolursamith/aquahealth.git'
BRANCH = 'develop'                                            # the single development branch
REPO_DIR = pathlib.Path('/content/aquahealth')
if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git pull --ff-only
!git log -1 --oneline
!pip install -q -r requirements/experiments.txt   # torch/torchvision/opencv/matplotlib pins; nothing else

## 2. CUDA, GPU, VRAM, PyTorch version (abort here without a GPU)

`gpu_smoke.py` moves a tensor to the GPU and runs one forward/backward/optimizer step; `colab_preflight.py --env-only` reports Python, torch/torchvision vs the pins, CUDA version, GPU name, VRAM and the repository digests. Exit 2 = no CUDA.

In [ ]:
!python scripts/gpu_smoke.py --require-cuda
!python scripts/colab_preflight.py --env-only --require-cuda
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.get_device_name(0),
      '| VRAM GiB', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 3. Dataset: the five delivered folders on Drive → `data/raw/<key>` → local bundle (Layer 2 mechanism)

`MyDrive/AquaHealth/` holds the five source deliveries as delivered. The folder → key mapping is **not guessed**: `scripts/verify_drive_delivery.py` applies the committed mapping (`src/multi_dataset.py::DATASET_SOURCES`, the same rules as `scripts/link_raw_datasets.py`), counts images per key against the audited inventory (`clean_manifest.csv`: 3,503 / 133 / 454 / 2,137 / 1,208) and checks that every manifest row resolves to a file. `link_raw_datasets.py` then creates the `data/raw/<key>` symlinks — the runtime path-resolution layer; no manifest is touched.

Because Drive is a slow FUSE mount, `scripts/build_colab_bundle.py` copies exactly the 5,942 *included* images to the VM's local disk (`/content/aquahealth_data/aquahealth_bundle`), re-hashing each source file against the manifest on the way, and `AQUAHEALTH_COLAB_DATASET_ROOT` is pointed at that bundle. Raw files on Drive are only read.

In [ ]:
import os, pathlib
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DROP = pathlib.Path('/content/drive/MyDrive/AquaHealth')       # the five delivered folders
assert DRIVE_DROP.is_dir(), f'{DRIVE_DROP} not found'
!ls -la "{DRIVE_DROP}"
# 3a. map the visible folders to the five keys and compare with the audited inventory (exit 1 = STOP)
!python scripts/verify_drive_delivery.py --source "{DRIVE_DROP}"
# 3b. runtime path resolution only: data/raw/<key> -> Drive folders (symlinks; manifests unchanged)
!python scripts/link_raw_datasets.py --source "{DRIVE_DROP}" --force
!ls -la data/raw
# 3c. copy exactly the 5,942 included images to local disk, verifying every source SHA-256 against the manifest
DATA_ROOT = pathlib.Path('/content/aquahealth_data')
BUNDLE = DATA_ROOT / 'aquahealth_bundle'
if not (BUNDLE / 'bundle.sha256').is_file():
    !python scripts/build_colab_bundle.py --out "{BUNDLE}" --no-verify-hashes   # section 4 hashes every local copy
os.environ['AQUAHEALTH_COLAB_DATASET_ROOT'] = str(BUNDLE)
!python scripts/colab_dataset.py info
!python scripts/colab_dataset.py attach --force      # data/raw/<key> now -> the local bundle
!ls -la data/raw

## 4. Data integrity — 5,942 images by SHA-256, manifest / split / fold digests, frozen final test

`verify --hash all` hashes every attached image against the clean manifest and checks the bundle pins (SHA-256 of `clean_manifest.csv`, `development.csv`, `final_test.csv`, `folds.csv`), counts, canonical labels, group ids and fold ids. The second cell prints the committed digests of the frozen final test and the ten folds and confirms no final-test id is inside any fold manifest. A non-zero exit means **STOP** — never regenerate manifests on Colab.

In [ ]:
!python scripts/colab_dataset.py verify --hash all
!cat data/audit/split_v2/split_manifest.sha256; echo; cat data/audit/cv_v2/folds.sha256
import csv, hashlib, pathlib
sha = lambda p: hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
print('final_test.csv', sha('data/audit/split_v2/final_test.csv'))
test_ids = {r['image_id'] for r in csv.DictReader(open('data/audit/split_v2/final_test.csv'))}
for k in range(1, 11):
    for part in ('train', 'validation'):
        f = f'data/audit/cv_v2/fold_{k:02d}_{part}.csv'
        ids = {r['image_id'] for r in csv.DictReader(open(f))}
        assert not (ids & test_ids), f'{f} contains final-test ids'
        print(f'fold_{k:02d}_{part}: {len(ids):5d} rows  sha256 {sha(f)[:16]}  final-test overlap 0')
print('final test ids', len(test_ids), '— present in no fold manifest')

## 5. Persist `data/gan` and `results/v2` on Drive (recommended for the 10-fold run)

The full run takes roughly an hour; with the links below a disconnect loses nothing and `run_gan_all_folds.py` resumes at the first incomplete fold.

In [ ]:
PERSIST_ON_DRIVE = True
if PERSIST_ON_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = pathlib.Path('/content/drive/MyDrive/AquaHealth')
    for local, remote in (('results/v2', DRIVE / 'results_v2'), ('data/gan', DRIVE / 'gan')):
        remote.mkdir(parents=True, exist_ok=True)
        local = pathlib.Path(local)
        if local.exists() and not local.is_symlink():
            !rsync -a --ignore-existing "{local}/" "{remote}/"
            !rm -rf "{local}"
        if not local.is_symlink():
            local.parent.mkdir(parents=True, exist_ok=True)
            local.symlink_to(remote)
        print(local, '->', local.resolve())

## 6. GAN SMOKE TEST on CUDA — INFRASTRUCTURE CHECK, NOT A RESULT

`configs/gan_v2/smoke.json`: the same cDCGAN, 1 epoch on a seeded 256-image subset of fold 1's training rows, 8 + 8 synthetic images. Output goes to `data/gan/smoke/` and `results/v2/smoke/` — never to the real fold directories or the real registry. `--require-cuda` refuses CPU. Then `verify_gan_outputs.py --smoke` proves: files open as RGB 64×64, classes valid, manifest ↔ files, no validation / test id anywhere, registry digests.

In [ ]:
!rm -rf data/gan/smoke results/v2/smoke/registry.csv results/v2/smoke/GAN_MANIFEST.csv   # a smoke run is disposable
!python scripts/run_gan_fold.py --fold 1 --config configs/gan_v2/smoke.json --smoke --require-cuda
!python scripts/verify_gan_outputs.py --smoke --images all
!cat data/gan/smoke/fold_01/gan_run.json | head -40

## 7. Inspect the smoke images (they will be noisy — 1 epoch on 256 images)

In [ ]:
import csv, matplotlib.pyplot as plt
from PIL import Image
def show_grid(manifest_csv, title, n=16):
    rows = list(csv.DictReader(open(manifest_csv)))[:n]
    cols = min(8, len(rows)); nrows = (len(rows) + cols - 1) // cols
    fig, axes = plt.subplots(nrows, cols, figsize=(1.6 * cols, 1.8 * nrows))
    for ax, r in zip(axes.flat, rows):
        im = Image.open(r['filepath']); ax.imshow(im); ax.set_title(f"{r['unified_class'][:14]}\n{im.size[0]}x{im.size[1]} {im.mode}", fontsize=6); ax.axis('off')
    for ax in list(axes.flat)[len(rows):]: ax.axis('off')
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
show_grid('data/gan/smoke/fold_01/synthetic_manifest.csv', 'SMOKE fold 1 — 1 epoch / 256 images — not a result')

## 8. FULL GENERATION — one cDCGAN per fold, 10 folds (only after section 6 printed `RESULT: VERIFIED`)

`configs/gan_v2/default.json`: 30 epochs on the fold's full training rows (3,811–4,276 images), class-aware amounts (each class topped up to the fold's largest class, capped at 1× its real count; the largest class gets 0). Expected: ≈ 23,600 synthetic images over 10 folds. Resumable; a fold already complete under `data/gan/fold_XX/` is skipped. The script ends with `GAN_MANIFEST.csv`, `verification.json` and `generation_summary.json` under `results/v2/gan/`.

In [ ]:
RUN_FULL = True
if RUN_FULL:
    !mkdir -p results/v2/gan
    !python scripts/run_gan_all_folds.py --config configs/gan_v2/default.json --require-cuda --images all \
        2>&1 | tee -a results/v2/gan/run_gan_all_folds.console.log

## 9. Result validation: every fold verified, raw data unchanged

`verify_gan_outputs.py` (all PNGs opened and hashed) proves the seven result checks; `colab_dataset.py verify --hash sample` re-hashes a seeded sample of the raw images against the clean manifest after generation (no source image modified).

In [ ]:
!python scripts/verify_gan_outputs.py --images all
!python scripts/colab_dataset.py verify --hash sample
import json
s = json.load(open('results/v2/gan/generation_summary.json'))
print(json.dumps({k: v for k, v in s.items() if k != 'per_fold'}, indent=2))
for f in s['per_fold']:
    print(f"fold {f['fold']:2d} {f['status']:20s} synthetic {f.get('synthetic_images', '?'):>5}  train_s {f.get('train_seconds', '?')}  wall_s {f.get('wall_seconds_incl_decode_and_generation', '?')}")
!head -c 1200 results/v2/gan/registry.csv
for fold in (1, 5, 10):
    show_grid(f'data/gan/fold_{fold:02d}/synthetic_manifest.csv', f'fold {fold} — 30 epochs — first 16 synthetic images')

## 10. Export the records for the repository (`results/v2/gan/` is tracked; images/checkpoints are not)

Writes `aquahealth_gan_layer3_<commit>.tar.gz` to Drive: `results/v2/gan/*` (registry, GAN_MANIFEST.csv, verification.json, generation_summary.json, console log), every fold's `gan_run.json` / `summary.json` / `synthetic_manifest.csv` / `fold_XX_train_gan.csv`, the synthetic PNGs (~250 MB) and, if `INCLUDE_CHECKPOINTS`, the generators (~68 MB each). Locally: extract at the repository root, run `python scripts/verify_gan_outputs.py --images all`, commit `results/v2/gan/`.

In [ ]:
INCLUDE_CHECKPOINTS = True
commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
out = pathlib.Path(f'/content/drive/MyDrive/AquaHealth/aquahealth_gan_layer3_{commit}.tar.gz')
members = ['results/v2/gan']
for d in sorted(pathlib.Path('data/gan').resolve().glob('fold_*')):
    rel = pathlib.Path('data/gan') / d.name
    members += [str(rel / n) for n in ('gan_run.json', 'summary.json', 'synthetic_manifest.csv', f'{d.name}_train_gan.csv') if (d / n).is_file()]
    members += [str(rel / 'train')]
    if INCLUDE_CHECKPOINTS and (d / 'generator.pt').is_file():
        members.append(str(rel / 'generator.pt'))
!tar -czhf "{out}" {' '.join(members)}
!ls -la "{out}"